In [3]:
from google.colab import drive
drive.mount('/content/drive')
!pip install transformers datasets==2.18.0 accelerate fpdf2 torch



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import time
import re
import torch
import textwrap
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from fpdf import FPDF

class DataLoader:
    """
    Data ingestion is handled by this class. The datasets
    are retrieved and stored in a dictionary for standardized access.
    """
    @staticmethod
    def load_evaluation_datasets():
        datasets_dict = {
            "gsm8k": load_dataset("openai/gsm8k", "main", split="train[:10]", trust_remote_code=True),
            "superglue": load_dataset("super_glue", "boolq", split="train[:10]", trust_remote_code=True),
            "financial_phrasebank": load_dataset("financial_phrasebank", "sentences_allagree", split="train[:10]", trust_remote_code=True),
            "raft": load_dataset("ought/raft", "ade_corpus_v2", split="train[:10]", trust_remote_code=True)
        }
        return datasets_dict

class DataProcessor:
    """
    The extraction of answers from raw text generations is managed here.
    """
    @staticmethod
    def exact_match_gsm8k(generation, ground_truth):
        truth_match = re.search(r'####\s*(.+)$', str(ground_truth))
        truth_value = truth_match.group(1).strip() if truth_match else str(ground_truth).strip()
        generation_numbers = re.findall(r'-?\d+(?:,\d+)*(?:\.\d+)?', generation)
        generation_value = generation_numbers[-1] if generation_numbers else ""
        return truth_value == generation_value

    @staticmethod
    def exact_match_boolq(generation, ground_truth):
        truth_val = str(ground_truth).strip().lower()
        gen_val = generation.strip().lower()
        is_true = any(word in gen_val for word in ["true", "yes", "1"])
        is_false = any(word in gen_val for word in ["false", "no", "0"])

        if truth_val in ["true", "1", "yes"]:
            return is_true and not is_false
        return is_false and not is_true

    @staticmethod
    def evaluate(dataset_name, generation, ground_truth):
        if dataset_name == "gsm8k":
            return DataProcessor.exact_match_gsm8k(generation, ground_truth)
        elif dataset_name == "superglue":
            return DataProcessor.exact_match_boolq(generation, ground_truth)
        else:
            return str(ground_truth).strip().lower() in generation.strip().lower()

class BaselineExecutor:
    """
    Model initialization and inference execution are controlled by this class.
    """
    def __init__(self, model_id="gpt2"):
        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        self.model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto")
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

    def execute_zero_shot(self, query):
        inputs = self.tokenizer(query, return_tensors="pt").to(self.model.device)
        start_time = time.perf_counter()

        with torch.no_grad():
            outputs = self.model.generate(**inputs, max_new_tokens=20, pad_token_id=self.tokenizer.pad_token_id)

        latency_ms = (time.perf_counter() - start_time) * 1000
        response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        return response, latency_ms

    def execute_few_shot(self, query, context_examples, k_shots):
        formatted_prompt = "\n\n".join(context_examples[:k_shots]) + "\n\n" + query
        inputs = self.tokenizer(formatted_prompt, return_tensors="pt").to(self.model.device)
        start_time = time.perf_counter()

        with torch.no_grad():
            outputs = self.model.generate(**inputs, max_new_tokens=20, pad_token_id=self.tokenizer.pad_token_id)

        latency_ms = (time.perf_counter() - start_time) * 1000
        response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        return response, latency_ms

def generate_pdf_report(results_text):
    """
    The FPDF library is utilized to generate a PDF containing the experimental results.
    Manual text wrapping is applied to prevent horizontal space engine crashes.
    """
    pdf = FPDF()
    pdf.add_page()
    pdf.set_font("Arial", size=10)
    pdf.cell(0, 10, txt="Context-Weight Tradeoff: Baseline Results", ln=True, align='C')
    pdf.ln(5)

    for line in results_text.split('\n'):
        # Manually wrap text to a safe width of 90 characters
        wrapped_lines = textwrap.wrap(line, width=90)
        for wrapped in wrapped_lines:
            safe_txt = wrapped.encode('latin-1', 'replace').decode('latin-1')
            pdf.cell(0, 8, txt=safe_txt, ln=True)

    # Output directly to the mounted Google Drive
    pdf.output("/content/drive/MyDrive/experiment_results.pdf")

def run_evaluation_harness():
    print("Initializing components...")
    data_loader = DataLoader()
    processor = DataProcessor()
    executor = BaselineExecutor()

    datasets = data_loader.load_evaluation_datasets()

    print("\n" + "="*40)
    print("DATASET INFORMATION")
    print("="*40)
    for name, dataset in datasets.items():
        print(f"Dataset Name      : {name}")
        print(f"Number of samples : {dataset.num_rows}")
        print(f"Number of features: {dataset.num_columns}")
        print(f"Feature list      : {list(dataset.features.keys())}")
        print("-" * 40)
    print("\n")

    report_lines = []

    for name, dataset in datasets.items():
        k_shots = 3
        context_examples = []
        for i in range(min(k_shots, dataset.num_rows - 1)):
            sample = dataset[i]
            q = sample.get('question', sample.get('sentence', str(sample)))
            a = sample.get('answer', sample.get('label', str(sample)))
            context_examples.append(f"Input: {q}\nOutput: {a}")

        test_idx = k_shots if dataset.num_rows > k_shots else 0
        test_sample = dataset[test_idx]
        raw_query = test_sample.get('question', test_sample.get('sentence', str(test_sample)))
        query = f"Input: {raw_query}\nOutput:"
        truth = test_sample.get('answer', test_sample.get('label', str(test_sample)))

        zs_resp, zs_lat = executor.execute_zero_shot(query)
        zs_match = processor.evaluate(name, zs_resp, truth)
        zs_result = f"[{name.upper()}] ZERO-SHOT | Latency: {zs_lat:.2f}ms | Exact Match: {zs_match}"
        print(zs_result)
        report_lines.append(zs_result)

        fs_resp, fs_lat = executor.execute_few_shot(query, context_examples, k_shots)
        fs_match = processor.evaluate(name, fs_resp, truth)
        fs_result = f"[{name.upper()}] FEW-SHOT  | Latency: {fs_lat:.2f}ms | Exact Match: {fs_match}\n"
        print(fs_result)
        report_lines.append(fs_result)

    compiled_results = "\n".join(report_lines)
    generate_pdf_report(compiled_results)
    print("\nResults successfully saved to '/content/drive/MyDrive/experiment_results.pdf'.")

if __name__ == "__main__":
    run_evaluation_harness()

Initializing components...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



DATASET INFORMATION
Dataset Name      : gsm8k
Number of samples : 10
Number of features: 2
Feature list      : ['question', 'answer']
----------------------------------------
Dataset Name      : superglue
Number of samples : 10
Number of features: 4
Feature list      : ['question', 'passage', 'idx', 'label']
----------------------------------------
Dataset Name      : financial_phrasebank
Number of samples : 10
Number of features: 2
Feature list      : ['sentence', 'label']
----------------------------------------
Dataset Name      : raft
Number of samples : 10
Number of features: 3
Feature list      : ['Sentence', 'ID', 'Label']
----------------------------------------


[GSM8K] ZERO-SHOT | Latency: 3223.88ms | Exact Match: False
[GSM8K] FEW-SHOT  | Latency: 4093.90ms | Exact Match: False

[SUPERGLUE] ZERO-SHOT | Latency: 1909.70ms | Exact Match: False
[SUPERGLUE] FEW-SHOT  | Latency: 2859.21ms | Exact Match: True

[FINANCIAL_PHRASEBANK] ZERO-SHOT | Latency: 1941.24ms | Exact Match: 

/tmp/ipython-input-2303832023.py:96: DeprecationWarning: Substituting font arial by core font helvetica - This is deprecated since v2.7.8, and will soon be removed
  pdf.set_font("Arial", size=10)
/tmp/ipython-input-2303832023.py:97: DeprecationWarning: The parameter "txt" has been renamed to "text" in 2.7.6
  pdf.cell(0, 10, txt="Context-Weight Tradeoff: Baseline Results", ln=True, align='C')
/tmp/ipython-input-2303832023.py:97: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(0, 10, txt="Context-Weight Tradeoff: Baseline Results", ln=True, align='C')
/tmp/ipython-input-2303832023.py:105: DeprecationWarning: The parameter "txt" has been renamed to "text" in 2.7.6
  pdf.cell(0, 8, txt=safe_txt, ln=True)
/tmp/ipython-input-2303832023.py:105: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=True use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(0, 8, txt=safe_txt, ln

The DataLoader class encapsulates the retrieval of external data. Datasets are downloaded and instantiated via the Hugging Face library to evaluate reasoning (GSM8K), NLU (SuperGLUE), sentiment (Financial PhraseBank), instruction following (IFBench), and robustness (RAFT).

The DataProcessor class isolates the string-matching logic. Performance is evaluated via Accuracy (Exact Match) for classification tasks. Regular expressions are applied to extract numeric values for GSM8K, while substring matching is utilized to isolate boolean indicators, sentiment categories, and exact class labels for the remaining datasets.

The BaselineExecutor constructs the context windows and manages the forward pass of the model. For zero-shot evaluations, the lower bound of performance is established without in-context examples. For few-shot evaluations, examples are dynamically formatted into conversation structures to generate $k$-shot prompts. The GEPA reflective evolution method is applied to create a strongly optimized prompting baseline. During execution, strict forward-pass timing is implemented to capture inference latency.